In [1]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

In [2]:
# =============================
# CONFIG
# =============================

INPUT_XLSX = "/content/sample_data/Extended_SL_2021_2025.xlsx"
SHEET_NAME = "10 SL"

OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGETS = ["Total Revenue", "COGS", "SG&A"]

BASE_FEATURES = [
    "Order Date",
    "Region",
    "Geo",
    "Country",
    "Item type",
    "Customer",
    "Raw Material",
    "Direct Labor",
    "Freight",
    "Storage",
    "Packaging",
    "Indirect Labor",
    "Rent & Utility",
    "Overhead"
]

RANDOM_STATE = 42
TEST_SIZE = 0.25


In [3]:
# =============================
# 1. LOAD MESSY EXCEL
# =============================

def load_financial_excel(path, sheet_name=SHEET_NAME):
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)

    header_row_idx = None

    for i in range(min(15, len(raw))):
        values = raw.iloc[i].astype(str).str.strip().tolist()
        if "Total Revenue" in values and "COGS" in values and "SG&A" in values:
            header_row_idx = i
            break

    if header_row_idx is None:
        raise ValueError("Could not find header row with Total Revenue, COGS and SG&A.")

    headers = raw.iloc[header_row_idx].tolist()
    df = raw.iloc[header_row_idx + 1:].copy()
    df.columns = headers

    df = df.dropna(axis=1, how="all").dropna(axis=0, how="all")
    df = df.loc[:, ~pd.isna(df.columns)]
    df.columns = [str(c).strip() for c in df.columns]

    if "Order Date" not in df.columns:
        raise ValueError("Order Date column is required for daily forecasting.")

    df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")

    for col in df.columns:
        if col not in ["Region", "Geo", "Country", "Item type", "Customer", "Order Date"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Order Date"] + TARGETS).reset_index(drop=True)

    return df


In [4]:
# =============================
# 2. DAILY FEATURE ENGINEERING
# =============================

def add_daily_date_features(df):
    df = df.copy()
    df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")

    df["Year"] = df["Order Date"].dt.year
    df["Month"] = df["Order Date"].dt.month
    df["Day"] = df["Order Date"].dt.day
    df["DayOfWeek"] = df["Order Date"].dt.dayofweek
    df["Quarter"] = df["Order Date"].dt.quarter
    df["IsMonthEnd"] = df["Order Date"].dt.is_month_end.astype(int)
    df["IsMonthStart"] = df["Order Date"].dt.is_month_start.astype(int)

    # Cyclic date encoding
    df["Month_Sin"] = np.sin(2 * np.pi * df["Month"] / 12)
    df["Month_Cos"] = np.cos(2 * np.pi * df["Month"] / 12)

    df["DayOfWeek_Sin"] = np.sin(2 * np.pi * df["DayOfWeek"] / 7)
    df["DayOfWeek_Cos"] = np.cos(2 * np.pi * df["DayOfWeek"] / 7)

    return df


def create_daily_dataset(df):
    df = df.copy()
    df = add_daily_date_features(df)

    numeric_sum_cols = [
        "Raw Material",
        "Direct Labor",
        "Freight",
        "Storage",
        "Packaging",
        "Indirect Labor",
        "Rent & Utility",
        "Overhead",
        "Total Revenue",
        "COGS",
        "SG&A"
    ]

    numeric_sum_cols = [c for c in numeric_sum_cols if c in df.columns]

    categorical_cols = ["Region", "Geo", "Country", "Item type", "Customer"]
    categorical_cols = [c for c in categorical_cols if c in df.columns]

    date_cols = [
        "Year",
        "Month",
        "Day",
        "DayOfWeek",
        "Quarter",
        "IsMonthEnd",
        "IsMonthStart",
        "Month_Sin",
        "Month_Cos",
        "DayOfWeek_Sin",
        "DayOfWeek_Cos"
    ]

    aggregation = {}

    for col in numeric_sum_cols:
        aggregation[col] = "sum"

    for col in categorical_cols:
        aggregation[col] = lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0]

    for col in date_cols:
        aggregation[col] = "first"

    daily_df = (
        df.groupby("Order Date", as_index=False)
        .agg(aggregation)
        .sort_values("Order Date")
        .reset_index(drop=True)
    )

    return daily_df


In [5]:
# =============================
# 3. QUANTUM FEATURE MAP
# =============================

class QuantumFeatureMapTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_qubits=6, entanglement=True, random_state=42):
        self.n_qubits = n_qubits
        self.entanglement = entanglement
        self.random_state = random_state

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        rng = np.random.default_rng(self.random_state)
        self.input_dim_ = X.shape[1]

        self.proj_ = rng.normal(
            0,
            1 / np.sqrt(max(1, self.input_dim_)),
            size=(self.input_dim_, self.n_qubits)
        )

        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)

        theta = X @ self.proj_
        theta = np.tanh(theta) * np.pi

        quantum_features = [
            np.cos(theta),
            np.sin(theta),
            np.cos(2 * theta)
        ]

        if self.entanglement:
            pair_features = []

            for i in range(self.n_qubits):
                j = (i + 1) % self.n_qubits

                pair_features.append(np.cos(theta[:, i] * theta[:, j]))
                pair_features.append(np.sin(theta[:, i] + theta[:, j]))

            quantum_features.append(np.column_stack(pair_features))

        return np.hstack(quantum_features)


class IdentityTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X


In [6]:
# =============================
# 4. MODEL BUILDING
# =============================

def build_preprocessor(df):
    feature_cols = [
        "Region",
        "Geo",
        "Country",
        "Item type",
        "Customer",
        "Raw Material",
        "Direct Labor",
        "Freight",
        "Storage",
        "Packaging",
        "Indirect Labor",
        "Rent & Utility",
        "Overhead",
        "Year",
        "Month",
        "Day",
        "DayOfWeek",
        "Quarter",
        "IsMonthEnd",
        "IsMonthStart",
        "Month_Sin",
        "Month_Cos",
        "DayOfWeek_Sin",
        "DayOfWeek_Cos"
    ]

    feature_cols = [c for c in feature_cols if c in df.columns]

    categorical_cols = [
        c for c in feature_cols
        if df[c].dtype == "object"
    ]

    numeric_cols = [
        c for c in feature_cols
        if c not in categorical_cols
    ]

    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", encoder)
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

    return preprocessor, feature_cols


def build_baseline_model(preprocessor):
    return Pipeline([
        ("preprocess", preprocessor),
        ("regressor", MultiOutputRegressor(LinearRegression()))
    ])


def build_hybrid_quantum_model(preprocessor):
    hybrid_features = FeatureUnion([
        ("classical_features", IdentityTransformer()),
        ("quantum_features", QuantumFeatureMapTransformer(
            n_qubits=6,
            entanglement=True,
            random_state=RANDOM_STATE
        ))
    ])

    model = Pipeline([
        ("preprocess", preprocessor),
        ("hybrid_feature_layer", hybrid_features),
        ("scaler_after_quantum", StandardScaler()),
        ("regressor", MultiOutputRegressor(
            RidgeCV(alphas=np.logspace(-4, 4, 30))
        ))
    ])

    return model


In [7]:
# =============================
# 5. EVALUATION
# =============================

def evaluate_model(model, X_test, y_test, model_name):
    preds = model.predict(X_test)

    rows = []

    for i, target in enumerate(TARGETS):
        mae = mean_absolute_error(y_test.iloc[:, i], preds[:, i])
        rmse = np.sqrt(mean_squared_error(y_test.iloc[:, i], preds[:, i]))
        r2 = r2_score(y_test.iloc[:, i], preds[:, i])

        rows.append({
            "Model": model_name,
            "Target": target,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    return pd.DataFrame(rows), preds

In [8]:
# =============================
# 6. TRAIN AND SAVE DAILY MODEL
# =============================

def train_and_save_adaptive_daily_model(input_xlsx=INPUT_XLSX):
    raw_df = load_financial_excel(input_xlsx)
    daily_df = create_daily_dataset(raw_df)

    print("Daily dataset shape:", daily_df.shape)

    preprocessor, feature_cols = build_preprocessor(daily_df)

    X = daily_df[feature_cols].copy()
    y = daily_df[TARGETS].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True
    )

    baseline = build_baseline_model(preprocessor)
    quantum = build_hybrid_quantum_model(preprocessor)

    baseline.fit(X_train, y_train)
    quantum.fit(X_train, y_train)

    baseline_metrics, baseline_preds = evaluate_model(
        baseline,
        X_test,
        y_test,
        "Daily Baseline Multiple Linear Regression"
    )

    quantum_metrics, quantum_preds = evaluate_model(
        quantum,
        X_test,
        y_test,
        "Daily Hybrid Quantum Ridge Regression"
    )

    metrics = pd.concat([baseline_metrics, quantum_metrics], ignore_index=True)

    print("\nModel Comparison:")
    print(metrics)

    best_model_per_target = {}

    final_preds = np.zeros_like(baseline_preds)

    for i, target in enumerate(TARGETS):
        base_r2 = baseline_metrics.loc[
            baseline_metrics["Target"] == target, "R2"
        ].values[0]

        quantum_r2 = quantum_metrics.loc[
            quantum_metrics["Target"] == target, "R2"
        ].values[0]

        if quantum_r2 > base_r2:
            best_model_per_target[target] = "quantum"
            final_preds[:, i] = quantum_preds[:, i]
        else:
            best_model_per_target[target] = "baseline"
            final_preds[:, i] = baseline_preds[:, i]

    print("\nBest model selected per target:")
    print(best_model_per_target)

    final_rows = []

    for i, target in enumerate(TARGETS):
        mae = mean_absolute_error(y_test.iloc[:, i], final_preds[:, i])
        rmse = np.sqrt(mean_squared_error(y_test.iloc[:, i], final_preds[:, i]))
        r2 = r2_score(y_test.iloc[:, i], final_preds[:, i])

        final_rows.append({
            "Model": f"Adaptive Best Model ({best_model_per_target[target]})",
            "Target": target,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    final_metrics = pd.DataFrame(final_rows)

    print("\nFinal Adaptive Model Performance:")
    print(final_metrics)

    model_package = {
        "baseline_model": baseline,
        "quantum_model": quantum,
        "best_model_per_target": best_model_per_target,
        "targets": TARGETS,
        "features": feature_cols,
        "note": "Adaptive daily model. Uses quantum only when it improves validation performance."
    }

    model_path = OUTPUT_DIR / "adaptive_daily_hybrid_financial_model.pkl"
    metrics_path = OUTPUT_DIR / "adaptive_daily_model_metrics.csv"

    joblib.dump(model_package, model_path)

    all_metrics = pd.concat(
        [metrics, final_metrics],
        ignore_index=True
    )

    all_metrics.to_csv(metrics_path, index=False)

    print("\nSaved adaptive model:")
    print(model_path)
    print(metrics_path)

    return model_package, all_metrics

In [9]:
# =============================
# 7. USER INPUT CONVERSION
# =============================

def prepare_user_daily_input(user_df, required_features):
    user_df = user_df.copy()

    if "Order Date" not in user_df.columns:
        raise ValueError("User input must contain Order Date column.")

    user_df["Order Date"] = pd.to_datetime(user_df["Order Date"], errors="coerce")

    if user_df["Order Date"].isna().any():
        raise ValueError("Invalid Order Date found in user input.")

    user_df = add_daily_date_features(user_df)

    for col in required_features:
        if col not in user_df.columns:
            user_df[col] = np.nan

    return user_df[required_features]


def predict_adaptive_daily_from_user_input(
    user_input_df,
    model_pkl="model_outputs/adaptive_daily_hybrid_financial_model.pkl"
):
    package = joblib.load(model_pkl)

    baseline_model = package["baseline_model"]
    quantum_model = package["quantum_model"]
    best_model_per_target = package["best_model_per_target"]
    features = package["features"]
    targets = package["targets"]

    prepared_input = prepare_user_daily_input(user_input_df, features)

    baseline_preds = baseline_model.predict(prepared_input)
    quantum_preds = quantum_model.predict(prepared_input)

    final_preds = np.zeros_like(baseline_preds)

    for i, target in enumerate(targets):
        if best_model_per_target[target] == "quantum":
            final_preds[:, i] = quantum_preds[:, i]
        else:
            final_preds[:, i] = baseline_preds[:, i]

    output = user_input_df[["Order Date"]].copy()

    for i, target in enumerate(targets):
        output[f"Predicted Daily {target}"] = final_preds[:, i]
        output[f"Model Used for {target}"] = best_model_per_target[target]

    return output

In [10]:

# =============================
# 8. FUTURE DAILY FORECAST
# =============================

def create_future_daily_input(
    start_date,
    end_date,
    region,
    geo,
    country,
    item_type,
    customer,
    raw_material,
    direct_labor,
    freight,
    storage,
    packaging,
    indirect_labor,
    rent_utility,
    overhead
):
    dates = pd.date_range(start=start_date, end=end_date, freq="D")

    future_df = pd.DataFrame({
        "Order Date": dates,
        "Region": region,
        "Geo": geo,
        "Country": country,
        "Item type": item_type,
        "Customer": customer,
        "Raw Material": raw_material,
        "Direct Labor": direct_labor,
        "Freight": freight,
        "Storage": storage,
        "Packaging": packaging,
        "Indirect Labor": indirect_labor,
        "Rent & Utility": rent_utility,
        "Overhead": overhead
    })

    return future_df


In [11]:
train_and_save_adaptive_daily_model(INPUT_XLSX)

    # Example user input for future daily forecast
future_input = create_future_daily_input(
        start_date="2026-01-01",
        end_date="2026-01-10",
        region="Asia",
        geo="APAC",
        country="India",
        item_type="Office Supplies",
        customer="Customer A",
        raw_material=50000,
        direct_labor=12000,
        freight=4000,
        storage=2500,
        packaging=1500,
        indirect_labor=8000,
        rent_utility=5000,
        overhead=7000
    )
daily_forecast = predict_adaptive_daily_from_user_input(future_input)

print("\nFuture Daily Forecast:")
print(daily_forecast)

daily_forecast.to_csv(
        OUTPUT_DIR / "future_daily_forecast.csv",
        index=False
    )


Daily dataset shape: (65, 28)

Model Comparison:
                                       Model         Target           MAE  \
0  Daily Baseline Multiple Linear Regression  Total Revenue  1.901511e+04   
1  Daily Baseline Multiple Linear Regression           COGS  6.890760e-10   
2  Daily Baseline Multiple Linear Regression           SG&A  7.488730e+03   
3      Daily Hybrid Quantum Ridge Regression  Total Revenue  5.137934e+04   
4      Daily Hybrid Quantum Ridge Regression           COGS  9.546177e+03   
5      Daily Hybrid Quantum Ridge Regression           SG&A  2.873825e+04   

           RMSE        R2  
0  2.361902e+04  0.988050  
1  8.485959e-10  1.000000  
2  9.573098e+03  0.953650  
3  6.898606e+04  0.898058  
4  1.250148e+04  0.995241  
5  4.147927e+04  0.129820  

Best model selected per target:
{'Total Revenue': 'baseline', 'COGS': 'baseline', 'SG&A': 'baseline'}

Final Adaptive Model Performance:
                            Model         Target           MAE          RMSE 